# Category generation

Run **all cells** with the working directory set to the **repository root** (the folder that contains `pyproject.toml`).

This notebook:

- Defines property groups for **NPS Category**, **NPS Value Category**, **Has Numeric NPS**, and combined **NPS All**.
- Builds few-shot examples automatically when you pass `csv_path=...` and omit `examples` (see `ClassificationCategory.__init__` and `nps_crawling.config.Config`: `CLASSIFICATION_FEW_SHOT_NUM_EXAMPLES`, `CLASSIFICATION_FEW_SHOT_TEXT_COLUMN`, `CLASSIFICATION_FEW_SHOT_SAMPLE_SEED`, plus the same split/seed as evaluation: `CLASSIFICATION_RANDOM_SEED`, `CLASSIFICATION_GROUND_TRUTH_TEST_SIZE`).
- Instantiating each category writes JSON under `src/nps_crawling/classification/configurations/categories/<name>/<stable_id>.json` when that file does not exist yet.
- Pass `num_examples=` on a category to override the config default for that category only.
- The last cell syncs **NPS All** into `NPS_ALL_WITH_QWEN.json`.

**CLI:** From the repo root run python train/regenerate_category_configs.py to delete existing JSON in the NPS category folders (and TestCat), regenerate categories, and refresh NPS_ALL_WITH_QWEN.json.

**Tip:** To change how many few-shots all categories use by default, edit `Config.CLASSIFICATION_FEW_SHOT_NUM_EXAMPLES` in `config.py` (or set `num_examples=None` everywhere below to rely purely on config).

In [3]:
# NPS Category Creation
from __future__ import annotations

import json
from pathlib import Path

from nps_crawling.classification.categories.category import (
    ClassificationCategory,
    ClassificationProperty,
    ClassificationType,
)

nps_category_properties = [
    ClassificationProperty(
        name="KPI_CURRENT_VALUE",
        description="The text reports a specific NPS value.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="KPI_TREND",
        description="The text describes change over time.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="KPI_HISTORICAL_COMPARISON",
        description="The text explicitly compares NPS against a previous period.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="TARGET_OUTLOOK",
        description="The text expresses a future goal, target, ambition, forecast, or expected future NPS.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="NPS_GOAL_REACHED",
        description="The text states that an NPS target or objective has been met or exceeded.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="METHODOLOGY_DEFINITION",
        description="The text explains what NPS is, how it is calculated, or what it measures.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="QUALITATIVE_ONLY",
        description="NPS is mentioned meaningfully but none of the other labels apply.",
        type=ClassificationType.BOOLEAN,
    ),
]

nps_value_category_properties = [
    ClassificationProperty(
        name="nps_value_fix",
        description="Direct NPS value.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_competition_industry",
        description="Industry benchmark or competitor NPS value.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_value_over",
        description="Threshold value associated with above, over, greater than, more than, or at least.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_value_below",
        description="Threshold value associated with below, under, less than, or at most.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_goal_value",
        description="Explicit NPS target value, only if value is at least 20.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_goal_change",
        description="Planned improvement amount such as improve by 10, increase by 5, or raise by 7.",
        type=ClassificationType.FLOAT,
    ),
]

has_numeric_nps_property = ClassificationProperty(
    name="has_numeric_nps",
    description="True if the text contains at least one explicit numeric NPS value.",
    type=ClassificationType.BOOLEAN,
)

all_classification_properties = [
    *nps_category_properties,
    *nps_value_category_properties,
    has_numeric_nps_property,
]

prompt_base = """You are an expert at analyzing text to identify and extract information about Net Promoter Score (NPS). NPS is a widely used metric that measures customer loyalty and satisfaction by asking customers how likely they are to recommend a product or service to others. Your task is to read the provided text and determine whether it contains specific information related to NPS, as defined by the following properties:"""

prompt_base_nps_category = (
    prompt_base
    + " Focus on the multi-label boolean dimensions only (whether each statement type applies)."
)
prompt_base_nps_value = (
    prompt_base
    + " Focus on extracting numeric NPS-related values only (direct value, benchmarks, thresholds, goals)."
)
prompt_base_has_numeric = (
    prompt_base
    + " Answer only whether the text states at least one concrete numeric NPS value."
)

In [1]:
from nps_crawling.config import Config
import pandas as pd

all_snippets = str((Config.ROOT_DIR / "train" / "ground_truth_final.csv").resolve())
example_snippets = str((Config.ROOT_DIR / "train" / "few_shot_examples.csv").resolve())

all_df = pd.read_csv(all_snippets)
example_df = pd.read_csv(example_snippets)

example_texts = set(
    example_df["snippet_text_short"].dropna().astype(str)
)
filtered_df = all_df[
    ~all_df["snippet_text_short"].fillna("").astype(str).isin(example_texts)
]

output_path = Config.ROOT_DIR / "train" / "ground_truth_final_without_examples.csv"
filtered_df.to_csv(output_path, index=False)

df = filtered_df
print(f"Original rows: {len(all_df)}")
print(f"Filtered rows: {len(filtered_df)}")
print(f"Saved to: {output_path}")

Original rows: 703
Filtered rows: 697
Saved to: C:\Users\silas\VSProjects\NPS-Crawling\train\ground_truth_final_without_examples.csv


### Few-shot counts

Omit `num_examples` to use `Config.CLASSIFICATION_FEW_SHOT_NUM_EXAMPLES`. The combined **NPS All** task uses fewer examples below to keep prompts smaller.

In [4]:
path = r"train\ground_truth_final_without_examples.csv"
categories = {
    "NPS Category": ClassificationCategory(
        "NPS Category",
        nps_category_properties,
        prompt_base_nps_category,
        csv_path=path,
        num_examples=0
    ),
    "NPS Value Category": ClassificationCategory(
        "NPS Value Category",
        nps_value_category_properties,
        prompt_base_nps_value,
        csv_path=path,
        num_examples=0
    ),
    "Has Numeric NPS": ClassificationCategory(
        "Has Numeric NPS",
        [has_numeric_nps_property],
        prompt_base_has_numeric,
        csv_path=path,
        num_examples=0,
    ),
    "NPS All": ClassificationCategory(
        "NPS All",
        all_classification_properties,
        prompt_base,
        csv_path=path,
        num_examples=0,
    ),
}

for label, cat in categories.items():
    print(
        f"{label}:\n  {cat.config_path}\n"
        f"  num_examples_field={cat.num_examples!r}  len(examples)={len(cat.examples)}  stable_id={cat.stable_id}\n"
    )

NPS Category:
  C:\Users\silas\VSProjects\NPS-Crawling\src\nps_crawling\classification\configurations\categories\NPS Category\f29fd305959bd3735b80fb2c7c75e6ea7f384bc9cafc442b41e45bfd667b2efe.json
  num_examples_field=0  len(examples)=0  stable_id=f29fd305959bd3735b80fb2c7c75e6ea7f384bc9cafc442b41e45bfd667b2efe

NPS Value Category:
  C:\Users\silas\VSProjects\NPS-Crawling\src\nps_crawling\classification\configurations\categories\NPS Value Category\ab7516891d5dc0197d793715c8dc150e6cd2c4956e310fc51dd1e23dd1c56b64.json
  num_examples_field=0  len(examples)=0  stable_id=ab7516891d5dc0197d793715c8dc150e6cd2c4956e310fc51dd1e23dd1c56b64

Has Numeric NPS:
  C:\Users\silas\VSProjects\NPS-Crawling\src\nps_crawling\classification\configurations\categories\Has Numeric NPS\c1834f33598d154937eba5e11142b5d108e1a7a351d7bd9329ad6a19624ae721.json
  num_examples_field=0  len(examples)=0  stable_id=c1834f33598d154937eba5e11142b5d108e1a7a351d7bd9329ad6a19624ae721

NPS All:
  C:\Users\silas\VSProjects\NPS-Cr

In [ ]:
from nps_crawling.classification.models.model import ground_truth_row_to_example
from nps_crawling.config import Config
text_column = Config.CLASSIFICATION_FEW_SHOT_TEXT_COLUMN
train_df = pd.read_csv(example_snippets)
train_df = train_df.dropna(subset=[text_column])
train_df = train_df[train_df[text_column].astype(str).str.strip() != ""]
for name, category in categories.items():
    examples = [ground_truth_row_to_example(row, text_column, category) for _, row in train_df.iterrows()]
    print(len(examples))
    category.examples = examples
    new_category = ClassificationCategory.from_dict(category.to_dict())

6
6
6
6


: 